## Agenda

- In-Code Exemplars (Few-Shot Learning within context)

- Instructional Scaffolding (Step-by-Step Guidance)

- Contextual Priming with constraints and roles

- Pattern Seeding and Iterative Refinement

- Negative prompts to exclude unwanted suggestions

In [ ]:
import langchain
print(langchain.__version__) ## check your version

0.3.27


In [ ]:
!pip install -q -U google-generativeai langchain_openai langchain_google_genai langchain ## install required libraries

In [ ]:
## defining function to display response of LLM
import pathlib
import textwrap
import getpass

import google.generativeai as genai
from langchain_google_genai import ChatGoogleGenerativeAI

# Used to securely store your API key
from google.colab import userdata

from IPython.display import display
from IPython.display import Markdown

## function to display response of LLM
def to_markdown(text):
  text = text.replace('•', '  *')
  return Markdown(textwrap.indent(text, '> ', predicate=lambda _: True))

In [ ]:
## LLM API key setup
import os
from google.colab import userdata
os.environ["BOSCH_API_KEY"] = userdata.get('BOSCH_API_KEY')

#os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

Let's break down each advanced prompting pattern conceptually, and understand through code examples.

This will help you master these patterns quickly and practically, focusing on how they work in the LangChain ecosystem.

## 1. In-Code Exemplars (Few-Shot Learning within Context)

**Concept:**  
Few-shot learning means providing the model with a few examples ("shots") of the desired input-output behavior directly in the prompt, so it learns the pattern and generalizes to new cases.

**Example:**  
Suppose you want Gemini to convert sentences to polite requests.

**Code:**
```python
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(model="gemini-1.5-flash", temperature=0.2)

prompt = """
Convert the following sentences into polite requests.

Example 1:
Input: Give me the report.
Output: Could you please give me the report?

Example 2:
Input: Send the file now.
Output: Would you mind sending the file now?

Now convert:
Input: Close the door.
Output:
"""

response = llm.invoke(prompt)
print(response.content)
```
This uses in-prompt exemplars to teach the model the desired transformation.



In [ ]:
import os
import requests
import json

class BoschGeminiLite:
    """
    Minimal wrapper that mirrors your working requests.post payload
    for the 'google-gemini-2-0-flash-lite' deployment.
    """
    def __init__(
        self,
        api_key: str | None = None,

        base_url: str = "https://aoai-farm.bosch-temp.com/api/openai",
        deployment: str = "google-gemini-2-0-flash-lite",
        body_model_value: str = "gemini-2.0-flash-lite"  # <- keep EXACTLY as in your working snippet
    ):
        self.api_key = api_key or os.getenv("BOSCH_API_KEY")
        if not self.api_key:
            raise ValueError("Set BOSCH_API_KEY or pass api_key=...")

        self.url = f"{base_url}/deployments/{deployment}/chat/completions"
        self.headers = {
            "Content-Type": "application/json",
            "genaiplatform-farm-subscription-key": self.api_key,
        }

        self.body_model_value = body_model_value  # exact "model" value required by farm

    def invoke(self, prompt: str, max_tokens: int = 512, temperature: float = 0.7):
        """
        Sends the exact JSON body you used in your working .py file.
        """
        data = {
            "model": self.body_model_value,  # keep this EXACTLY like your working request
            "messages": [
                {"role": "user", "content": prompt}
            ],
            # (Optional: include other fields you know work in your environment)
            # "max_tokens": max_tokens,
            # "temperature": temperature,
        }

        resp = requests.post(self.url, headers=self.headers, json=data)
        resp.raise_for_status()

        payload = resp.json()
        try:
            return payload["choices"][0]["message"]["content"]
        except Exception:
            # Fall back to raw JSON if structure changes
            return json.dumps(payload, indent=2)


In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

#llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash-lite", temperature=0.2) ## or try gemini-1.5-flash

llm = BoschGeminiLite(                      # set None if not needed
)


prompt = """
Convert the following sentences into polite requests.

Example 1:
Input: Give me the report.
Output: Could you please give me the report?

Example 2:
Input: Send the file now.
Output: Would you mind sending the file now?

Now convert:
Input: Close the door.
Output:
"""

response = llm.invoke(prompt)
to_markdown(response)

> Could you please close the door?


## 2. Instructional Scaffolding (Step-by-Step Guidance)

**Concept:**  
Guide the model through a complex task by breaking it into clear, sequential steps.

**Example:**  
Summarize a news article in three steps: identify the topic, list key facts, then write a summary.

**Code:**
```python
article = "LangChain's architecture is modular, consisting of layers that include core abstractions, integrations, and composable components, as illustrated in its conceptual diagrams. A typical diagram shows LangChain as a layered system: at the base is langchain-core, which defines interfaces for LLMs, vector stores, and tools; above that, the main langchain package handles chains and agents for workflow orchestration; and on top, integration packages connect to external services"

prompt = f"""
Summarize the following article in three steps:
1. Identify the main topic.
2. List key facts.
3. Write a concise summary.

Article: {article}
"""

response = llm.invoke(prompt)
print(response.content)
```
This scaffolds the reasoning, leading to more structured and reliable outputs.



In [ ]:
article = """LangChain's architecture is modular, consisting of layers that include core abstractions, integrations,
and composable components, as illustrated in its conceptual diagrams. A typical diagram shows LangChain as a layered system:
at the base is langchain-core, which defines interfaces for LLMs, vector stores, and tools; above that, the main langchain package
handles chains and agents for workflow orchestration; and on top, integration packages connect to external services"""

prompt = f"""
Summarize the following article in three steps:
1. Identify the main topic.
2. List key facts.
3. Write a concise summary.

Article: {article}
"""

response = llm.invoke(prompt)
to_markdown(response)

> Okay, here's a three-step summary of the LangChain architecture article:
> 
> 1.  **Identify the main topic:** The architecture and structure of the LangChain framework.
> 
> 2.  **List key facts:**
>     *   LangChain has a modular architecture.
>     *   It's organized in layers, typically represented in diagrams.
>     *   The base layer is `langchain-core`, defining core interfaces (LLMs, vector stores, tools).
>     *   The main `langchain` package handles chains and agents for workflow orchestration.
>     *   Integration packages connect to external services.
> 
> 3.  **Write a concise summary:**  LangChain's architecture is modular and layered. At its core is `langchain-core`, defining fundamental interfaces, with the main `langchain` package for orchestration. Integration packages extend functionality.


## 3. Contextual Priming with Constraints and Roles

**Concept:**  
Set the model’s role and add constraints to guide its behavior and output.

**Example:**  
Ask Gemini to act as a *grammar teacher* and only correct grammar mistakes, not content.

**Code:**
```python
prompt = """
You are a strict grammar teacher. Only correct grammar mistakes in the following sentence, do not change the meaning or add new content.

Sentence: She go to school every day.
"""

response = llm.invoke(prompt)
print(response)
```

Role and constraints are explicitly set, priming the model for focused output.

> **Priming an LLM** (Large Language Model) refers to providing the model with specific instructions, examples, or context before generating text. This technique helps guide the model's output, leading to more focused, relevant, and higher-quality results. **By carefully crafting the "priming" input, users can influence the style, tone, and subject matter of the generated content.**

In [ ]:
prompt = """
You are a strict grammar teacher. Only correct grammar mistakes in the following sentence, do not change the meaning or add new content.

Sentence: She go to school every day.
"""

response = llm.invoke(prompt)
print(response)

She **goes** to school every day.



## 4. Pattern Seeding and Iterative Refinement

**Concept:**  
Start with a broad prompt, then iteratively add specific constraints to refine the output.

**Example:**  
First, ask for a story. Then, add the constraint that it must be set in space and under 50 words.

**Code:**
```python
# First, broad prompt
response1 = llm.invoke("Write a short story.")
print("Broad:", response1)

# Refined prompt
prompt2 = """
Write a very short story set in space. The story must be under 50 words.
"""
response2 = llm.invoke(prompt2)
print("Refined:", response2)
```
You can keep adding constraints in further iterations to shape the output.

## 5. Negative Prompts to Exclude Unwanted Suggestions

**Concept:**  
Explicitly instruct the model to avoid certain topics, styles, or content.

**Example:**  
Generate a product description for a toy, but *do not* mention age or safety.

**Code:**
```python
prompt = """
Write a product description for a new toy. Do not mention age recommendations or safety information.

Product: RoboDog, a robot dog that can bark and fetch.
"""

response = llm.invoke(prompt)
print(response)
```
Negative prompts help filter out unwanted or irrelevant information.


You can combine these prompting patterns for even more powerful and controlled outputs. Experiment with chaining them together in your LangChain apps for best results.

# Playground ( Try-outs for reinforcement learning )

Participants are expected to copy the code block, make changes according to one's functional department and experiment.

## 1. In-Code Exemplars (Few-Shot Learning within Context)

**Prompt:**  
Generate code comments explaining AI functions in automotive software.

```python
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(model="gemini-pro", temperature=0.2)

prompt = """
Add a code comment that explains the purpose of each AI function in automotive software.

Example 1:
def lane_detection(image):
    # Detects lane markings in the input image for autonomous driving.

Example 2:
def pedestrian_alert(sensor_data):
    # Alerts the driver when a pedestrian is detected nearby.

Now add a comment:
def adaptive_cruise_control(speed, distance):
"""

response = llm.invoke(prompt)
print(response)
```

## 2. Instructional Scaffolding (Step-by-Step Guidance)

**Prompt:**  
Explain how GenAI could improve automotive predictive maintenance, step by step.

```python
article = "Automotive companies are integrating GenAI to predict vehicle component failures before they occur."

prompt = f"""
Explain in three steps how GenAI can enhance predictive maintenance in the automotive industry:
1. Identify the main challenge.
2. Describe how GenAI addresses it.
3. Summarize the expected benefits.

Article: {article}
"""

response = llm.invoke(prompt)
print(response)
```

## 3. Contextual Priming with Constraints and Roles

**Prompt:**  
Act as an automotive software architect and summarize the impact of AI on vehicle safety—limit to technical aspects only.

```python
prompt = """
You are an automotive software architect. Summarize the impact of AI on vehicle safety, focusing only on technical aspects such as sensor fusion, real-time data analysis, and fail-safe mechanisms. Do not discuss regulatory or business perspectives.
"""

response = llm.invoke(prompt)
print(response)
```

## 4. Pattern Seeding and Iterative Refinement

**Prompt:**  
Start broad, then refine: first, describe AI applications in automotive; then, add a constraint to focus on electric vehicles.

```python
# Broad prompt
response1 = llm.invoke("Describe the main applications of AI in the automotive industry.")
print("Broad:", response1)

# Refined prompt
prompt2 = """
Describe the main applications of AI specifically in electric vehicles, such as battery management, range prediction, and smart charging.
"""
response2 = llm.invoke(prompt2)
print("Refined:", response2)
```

## 5. Negative Prompts to Exclude Unwanted Suggestions

**Prompt:**  
Generate a list of AI use cases in automotive manufacturing, but do not mention autonomous driving.

```python
prompt = """
List five innovative AI use cases in automotive manufacturing. Do not include autonomous driving or self-driving features.
"""

response = llm.invoke(prompt)
print(response)
```

In [ ]:
## 1. In-Code Exemplars (Few-Shot Learning within Context)

from langchain_google_genai import ChatGoogleGenerativeAI

#llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite", temperature=0.2)

prompt = """
Add a code comment that explains the purpose of each AI function in automotive software.

Example 1:
def lane_detection(image):
    # Detects lane markings in the input image for autonomous driving.

Example 2:
def pedestrian_alert(sensor_data):
    # Alerts the driver when a pedestrian is detected nearby.

Now add a comment:
def adaptive_cruise_control(speed, distance):
"""

response = llm.invoke(prompt)
print(response)

```python
def adaptive_cruise_control(speed, distance):
    # Adjusts vehicle speed to maintain a safe distance from the vehicle ahead.
```



In [ ]:
## 2. Instructional Scaffolding (Step-by-Step Guidance)

article = "Automotive companies are integrating GenAI to predict vehicle component failures before they occur."

prompt = f"""
Explain in three steps how GenAI can enhance predictive maintenance in the automotive industry:
1. Identify the main challenge.
2. Describe how GenAI addresses it.
3. Summarize the expected benefits.

Article: {article}
"""

response = llm.invoke(prompt)
#print(response.content)
to_markdown(response)

> Here's a breakdown of how GenAI enhances predictive maintenance in the automotive industry, following your three-step structure:
> 
> **1. Identify the Main Challenge:**
> 
> *   The main challenge is **proactively predicting component failures** in vehicles. This involves:
>     *   Identifying which components are most likely to fail.
>     *   Determining the timeframe of potential failures.
>     *   Reducing the cost and risk associated with unexpected breakdowns.
> 
> **2. Describe How GenAI Addresses It:**
> 
> *   GenAI tackles this challenge through advanced data analysis and pattern recognition. Specifically:
>     *   **Data Ingestion and Processing:** GenAI algorithms ingest vast amounts of data from various sources:
>         *   Vehicle sensors (engine temperature, oil pressure, tire pressure, etc.)
>         *   Historical maintenance records (repairs, replacements)
>         *   Manufacturing data (component specifications, build dates)
>         *   External data (weather conditions, driving habits)
>     *   **Pattern Recognition and Anomaly Detection:** GenAI models are trained to identify complex patterns and anomalies within this data. This allows them to:
>         *   Detect subtle deviations from normal operating conditions, which may indicate early-stage component degradation.
>         *   Correlate different data points to pinpoint the root causes of potential failures.
>         *   Model component lifecycles and predict when failures are most likely to occur.
>     *   **Predictive Modeling and Actionable Insights:** Based on the analysis, GenAI generates:
>         *   Predictions about the likelihood and timing of component failures.
>         *   Recommendations for preventative maintenance actions (e.g., replacing a part, adjusting a setting).
>         *   Prioritization of maintenance tasks to optimize resource allocation.
> 
> **3. Summarize the Expected Benefits:**
> 
> *   The integration of GenAI for predictive maintenance yields several key benefits:
>     *   **Reduced Downtime:** Fewer unexpected breakdowns lead to less vehicle downtime and increased uptime.
>     *   **Lower Maintenance Costs:** Proactive maintenance based on predictions reduces the need for reactive repairs, minimizing repair expenses.
>     *   **Improved Vehicle Reliability:** Early detection of potential issues enhances vehicle reliability and safety.
>     *   **Enhanced Customer Satisfaction:** Predictive maintenance provides a better ownership experience for customers by mitigating unexpected breakdowns.
>     *   **Optimized Parts Inventory:** By predicting component failures, GenAI facilitates the efficient management of parts inventory, reducing waste and ensuring parts are available when needed.


In [ ]:
## 3. Contextual Priming with Constraints and Roles



In [ ]:
## 4. Pattern Seeding and Iterative Refinement



In [ ]:
## 5. Negative Prompts to Exclude Unwanted Suggestions

